# Embeddings Notebook (Core LLM Prep Pipeline)
Juan Sebastian Velandia

This notebook reuses the **core code ideas from Chapter 2** of *Build a Large Language Model From Scratch*: tokenization, sliding-window dataset creation, and token + positional embeddings.

Why this matters for LLMs and agentic systems:
- LLMs predict the next token, so text must be converted into token IDs and training windows.
- Agentic systems rely on LLMs that can preserve context across steps; good data windowing directly affects this context handling.
- Embeddings turn discrete symbols into trainable vectors where optimization can shape semantic structure.

In [1]:
from pathlib import Path
import importlib
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

print('tiktoken version:', importlib.metadata.version('tiktoken'))
print('torch version:', torch.__version__)

tiktoken version: 0.12.0
torch version: 2.9.1+cpu


## Step 1 — Tokenize raw text

Tokenization is the bridge between language and computation. A neural network cannot directly process words; it processes numbers.

For agentic systems, tokenization quality also affects tool calls, memory retrieval, and prompt boundaries because all those interactions are mediated by token sequences.

In [2]:
raw_text = Path('the-verdict.txt').read_text(encoding='utf-8')
tokenizer = tiktoken.get_encoding('gpt2')
enc_text = tokenizer.encode(raw_text)

print('Total characters:', len(raw_text))
print('Total tokens:', len(enc_text))
print('First 20 token IDs:', enc_text[:20])

Total characters: 20479
Total tokens: 5145
First 20 token IDs: [40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438]


## Step 2 — Build sliding-window training samples

An autoregressive LLM learns from pairs `(input_tokens, target_tokens)` where targets are shifted by one position.

Why this matters:
- It teaches next-token prediction, the core pretraining objective.
- Overlapping windows (small stride) expose the model to many local contexts, improving continuity and robustness.
- In agentic systems, this helps models maintain coherent multi-step reasoning over nearby context.

In [3]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={'<|endoftext|>'})
        assert len(token_ids) > max_length, 'Need at least max_length+1 tokens'

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding('gpt2')
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

In [4]:
dataloader = create_dataloader_v1(raw_text, batch_size=2, max_length=8, stride=4, shuffle=False)
inputs, targets = next(iter(dataloader))

print('inputs shape:', inputs.shape)
print('targets shape:', targets.shape)
print('First input row:', inputs[0])
print('First target row:', targets[0])

inputs shape: torch.Size([2, 8])
targets shape: torch.Size([2, 8])
First input row: tensor([  40,  367, 2885, 1464, 1807, 3619,  402,  271])
First target row: tensor([  367,  2885,  1464,  1807,  3619,   402,   271, 10899])


## Step 3 — Create token and positional embeddings

### Why do embeddings encode meaning, and how are they related to NN concepts?
Embeddings are trainable vectors looked up by token ID. During optimization, gradient descent adjusts these vectors so tokens used in similar predictive contexts end up with similar representations.

From a neural network perspective, an embedding layer is equivalent to selecting rows from a weight matrix (conceptually similar to one-hot input multiplied by a linear layer). The key is that the rows are **parameters** learned by backpropagation.

Meaning is therefore not hand-coded; it is an emergent geometric property of the learned parameter space, shaped by the loss function and data distribution.

In [5]:
vocab_size = 50257
output_dim = 256
max_length = 8

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(max_length, output_dim)

batch_inputs, _ = next(iter(create_dataloader_v1(raw_text, batch_size=4, max_length=max_length, stride=max_length, shuffle=False)))
token_embeddings = token_embedding_layer(batch_inputs)
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
input_embeddings = token_embeddings + pos_embeddings

print('batch_inputs shape:', batch_inputs.shape)
print('token_embeddings shape:', token_embeddings.shape)
print('pos_embeddings shape:', pos_embeddings.shape)
print('input_embeddings shape:', input_embeddings.shape)

batch_inputs shape: torch.Size([4, 8])
token_embeddings shape: torch.Size([4, 8, 256])
pos_embeddings shape: torch.Size([8, 256])
input_embeddings shape: torch.Size([4, 8, 256])


## Step 4 — Small experiment: effect of `max_length` and `stride`

We compare how many training samples are produced under different settings.

Expected behavior:
- Larger `max_length` → fewer windows because each sample consumes more tokens.
- Smaller `stride` → more overlap → more samples.

Overlap is useful because each token can be seen in multiple neighboring contexts, which improves local continuity learning. The tradeoff is extra compute and potentially more redundancy.

In [6]:
token_count = len(enc_text)
settings = [(32, 32), (32, 16), (64, 64), (64, 16), (128, 128), (128, 32)]

def count_samples(token_ids, max_length, stride):
    return len(range(0, len(token_ids) - max_length, stride))

print('Total tokens:', token_count)
print('max_length | stride | n_samples | overlap_ratio')
print('-' * 52)
for max_len, stride in settings:
    n = count_samples(enc_text, max_len, stride)
    overlap_ratio = 1 - (stride / max_len)
    print(f'{max_len:10d} | {stride:6d} | {n:9d} | {overlap_ratio:12.2f}')

Total tokens: 5145
max_length | stride | n_samples | overlap_ratio
----------------------------------------------------
        32 |     32 |       160 |         0.00
        32 |     16 |       320 |         0.50
        64 |     64 |        80 |         0.00
        64 |     16 |       318 |         0.75
       128 |    128 |        40 |         0.00
       128 |     32 |       157 |         0.75


## Experiment report (computed on this text)

Using this `the-verdict.txt` file (`5145` tokens), we get:
- `max_length=32, stride=32` → `160` samples
- `max_length=32, stride=16` → `320` samples
- `max_length=64, stride=64` → `80` samples
- `max_length=64, stride=16` → `318` samples
- `max_length=128, stride=128` → `40` samples
- `max_length=128, stride=32` → `157` samples

Interpretation:
- Halving stride (more overlap) can roughly double or quadruple samples depending on context length.
- Higher overlap increases training signal density for local transitions, which is often useful early in model development and for smaller datasets.
- For production-scale training, overlap is tuned to balance quality gains vs. compute cost.